In [1]:
%pip install "duckdb==1.5.6"

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ----------- ---------------------------- 3.7/13.2 MB 19.8 MB/s eta 0:00:01
   ----------------------------- ---------- 9.7/13.2 MB 26.3 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 23.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import csv
import shutil
import tempfile
import duckdb

FOLDER = Path(r"C:\Users\tahai\Desktop\EMM-Project\eda_outputs")

work = tempfile.TemporaryDirectory(prefix="emm_repair_")
con = duckdb.connect(str(Path(work.name) / "repair.duckdb"))

con.execute("SET memory_limit = '6GB'")
con.execute("SET threads = 4")
con.execute("SET preserve_insertion_order = false")

In [3]:
def sql_text(value):
    return "'" + str(value).replace("'", "''") + "'"


def csv_source(path):
    with path.open(newline="", encoding="utf-8-sig") as f:
        names = next(csv.reader(f))

    types = {
        name: (
            "VARCHAR" if name in ("subreddit", "user")
            else "DOUBLE" if name == "proportional_reciprocity"
            else "BIGINT"
        )
        for name in names
    }

    return (
        f"read_csv({sql_text(path.as_posix())}, "
        f"header=true, columns={types})"
    )


def check(source, expected_rows, unique=False):
    columns = [
        row[0]
        for row in con.execute(
            f"DESCRIBE SELECT * FROM {source}"
        ).fetchall()
    ]

    missing = " OR ".join(
        f'"{column}" IS NULL' for column in columns
    )

    result = con.execute(f"""
        SELECT
            count(*),
            count(DISTINCT subreddit),
            count(*) FILTER (WHERE {missing})
        FROM {source}
    """).fetchone()

    assert result == (expected_rows, 2046, 0), (
        f"Validation failed: {result}"
    )

    if unique:
        duplicate = con.execute(f"""
            SELECT 1
            FROM {source}
            GROUP BY subreddit, month, "user"
            HAVING count(*) > 1
            LIMIT 1
        """).fetchone()

        assert duplicate is None, "Duplicate observation keys."

    if "returned_next_month" in columns:
        invalid = con.execute(f"""
            SELECT count(*)
            FROM {source}
            WHERE returned_next_month NOT IN (0, 1)
        """).fetchone()[0]

        assert invalid == 0, "Invalid return labels."

    print(
        f"Verified: {result[0]:,} rows, "
        f"{result[1]:,} subreddits.",
        flush=True,
    )

In [4]:
sources = [
    ("activity", "user_activity.csv", 40_050_308),
    ("reciprocity", "user_reciprocity.csv", 40_050_308),
    ("labels", "user_return.csv", 36_031_393),
]

for table, filename, expected_rows in sources:
    print(f"Loading and checking {filename}...", flush=True)

    source = csv_source(FOLDER / filename)

    con.execute(f"""
        CREATE TABLE {table} AS
        SELECT * FROM {source}
    """)

    check(table, expected_rows, unique=True)

Loading and checking user_activity.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified: 40,050,308 rows, 2,046 subreddits.
Loading and checking user_reciprocity.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified: 40,050,308 rows, 2,046 subreddits.
Loading and checking user_return.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified: 36,031,393 rows, 2,046 subreddits.


In [5]:
print("Joining activity and reciprocity...", flush=True)

con.execute("""
    CREATE TABLE merged AS
    SELECT
        a.*,
        r.reciprocal_partners,
        r.outgoing_partners,
        r.proportional_reciprocity
    FROM activity a
    LEFT JOIN reciprocity r
    USING (subreddit, month, "user")
""")

check("merged", 40_050_308)

print("Adding next-month return labels...", flush=True)

con.execute("""
    CREATE TABLE final AS
    SELECT
        m.*,
        t.returned_next_month
    FROM labels t
    LEFT JOIN merged m
    USING (subreddit, month, "user")
""")

check("final", 36_031_393)

Joining activity and reciprocity...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified: 40,050,308 rows, 2,046 subreddits.
Adding next-month return labels...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified: 36,031,393 rows, 2,046 subreddits.


In [6]:
outputs = [
    ("merged", "user_activity_reciprocity.csv", 40_050_308),
    ("final", "reciprocity_return.csv", 36_031_393),
]

staged = []

for table, filename, expected_rows in outputs:
    new_path = FOLDER / (filename + ".new")

    print(f"Writing {filename}...", flush=True)

    con.execute(f"""
        COPY {table}
        TO {sql_text(new_path.as_posix())}
        (FORMAT CSV, HEADER TRUE)
    """)

    print("Checking exported file...", flush=True)
    check(csv_source(new_path), expected_rows)

    staged.append((new_path, FOLDER / filename))

Writing user_activity_reciprocity.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checking exported file...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified: 40,050,308 rows, 2,046 subreddits.
Writing reciprocity_return.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checking exported file...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified: 36,031,393 rows, 2,046 subreddits.


In [9]:
assert len(staged) == 2, "Both exports must pass validation."

con.close()

for new_path, original_path in staged:
    backup_path = original_path.with_name(
        original_path.name + ".incomplete.bak"
    )

    if original_path.exists() and not backup_path.exists():
        shutil.copy2(original_path, backup_path)

    new_path.replace(original_path)
    print("Replaced:", original_path.name)

work.cleanup()

print("Repair complete.")

Replaced: user_activity_reciprocity.csv
Replaced: reciprocity_return.csv
Repair complete.
